# LoRA Fine-Tuning Notebook For Agent Models

Use this notebook to fine-tune either:

1. `fundamental` agent using `data/gold/fundamental_finetune_data.jsonl`
2. `market_sentiment` agent using `data/gold/market_sentiment_finetune_data.jsonl`

This notebook contains the full fine-tuning code: JSONL loading, tokenization, LoRA adapter setup, training, saving, and inference test.

Run this on a GPU environment. On a normal laptop CPU, this will be very slow.

## 0. Install Dependencies

Run this cell once in your training environment. If you already installed these packages, skip it.

In [ ]:
# Uncomment if needed
# !pip install torch transformers datasets peft accelerate

## 1. Choose Agent And Model

Change `AGENT_NAME` to either `fundamental` or `market_sentiment`.

Use a base model that you can access from Hugging Face. For Mistral Small 4, replace `BASE_MODEL` with the exact allowed model id.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

# Choose one: 'fundamental' or 'market_sentiment'
AGENT_NAME = 'fundamental'

# Practical open model example. Replace with Mistral Small 4 id if you have access.
BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'

if AGENT_NAME == 'fundamental':
    TRAIN_JSONL = PROJECT_ROOT / 'data/gold/fundamental_finetune_data.jsonl'
    OUTPUT_DIR = PROJECT_ROOT / 'models/fundamental_lora'
elif AGENT_NAME == 'market_sentiment':
    TRAIN_JSONL = PROJECT_ROOT / 'data/gold/market_sentiment_finetune_data.jsonl'
    OUTPUT_DIR = PROJECT_ROOT / 'models/market_sentiment_lora'
else:
    raise ValueError('AGENT_NAME must be fundamental or market_sentiment')

MAX_LENGTH = 2048
EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 2e-4

print('Project root:', PROJECT_ROOT)
print('Agent:', AGENT_NAME)
print('Training JSONL:', TRAIN_JSONL)
print('Output adapter:', OUTPUT_DIR)

## 2. Create JSONL Fine-Tuning Data If Needed

If your agent-specific notebook already created the JSONL, this cell will skip. If not, it creates the file from current Gold labels and agent features.

In [ ]:
import csv

def label_from_score(score):
    score = float(score)
    if score <= 3.5:
        return 'LOW'
    if score <= 6.0:
        return 'MODERATE'
    return 'HIGH'

def load_gold_rows():
    gold_path = PROJECT_ROOT / 'data/gold/gold_risk_scores_ALL.csv'
    with open(gold_path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def create_fundamental_jsonl(path):
    from agents.fundamental_agent import FundamentalAnalysisAgent
    agent = FundamentalAnalysisAgent(silver_dir=str(PROJECT_ROOT / 'data/silver'))
    examples = []
    for row in load_gold_rows():
        ticker = row['ticker'].upper()
        try:
            result = agent.analyze(ticker)
        except Exception:
            continue
        target_score = float(row.get('fundamental_risk') or row['composite_risk'])
        target_label = label_from_score(target_score)
        user_payload = {
            'ticker': ticker,
            'task': 'Assess company fundamental risk from SEC-derived financial features.',
            'features': result.features,
            'evidence': result.evidence,
        }
        assistant_payload = {
            'fundamental_risk_score': round(target_score, 2),
            'fundamental_risk_label': target_label,
            'explanation': 'Use balance-sheet strength, profitability, leverage, and trend evidence to justify the score.'
        }
        examples.append({
            'messages': [
                {'role': 'system', 'content': 'You are a financial fundamental risk analyst. Return strict JSON.'},
                {'role': 'user', 'content': json.dumps(user_payload)},
                {'role': 'assistant', 'content': json.dumps(assistant_payload)},
            ]
        })
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for ex in examples:
            f.write(json.dumps(ex) + '\n')
    return len(examples)

def create_market_sentiment_jsonl(path):
    from agents.market_sentiment_agent import MarketSentimentAgent
    agent = MarketSentimentAgent(
        silver_dir=str(PROJECT_ROOT / 'data/silver'),
        news_path=str(PROJECT_ROOT / 'data/silver/silver_news_sentiment.csv'),
    )
    examples = []
    for row in load_gold_rows():
        ticker = row['ticker'].upper()
        try:
            result = agent.analyze(ticker)
        except Exception:
            continue
        volatility_score = float(row.get('volatility_risk') or row['composite_risk'])
        sentiment_score = float(row.get('sentiment_risk') or 5.0)
        target_score = round(volatility_score * 0.70 + sentiment_score * 0.30, 2)
        target_label = label_from_score(target_score)
        user_payload = {
            'ticker': ticker,
            'task': 'Assess market, volatility, and news sentiment risk.',
            'features': result.features,
            'evidence': result.evidence,
        }
        assistant_payload = {
            'market_sentiment_risk_score': target_score,
            'market_sentiment_risk_label': target_label,
            'explanation': 'Use volatility, beta, drawdown, price trend, and news sentiment evidence to justify the score.'
        }
        examples.append({
            'messages': [
                {'role': 'system', 'content': 'You are a market volatility and financial news sentiment risk analyst. Return strict JSON.'},
                {'role': 'user', 'content': json.dumps(user_payload)},
                {'role': 'assistant', 'content': json.dumps(assistant_payload)},
            ]
        })
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for ex in examples:
            f.write(json.dumps(ex) + '\n')
    return len(examples)

if TRAIN_JSONL.exists():
    print('JSONL already exists:', TRAIN_JSONL)
else:
    if AGENT_NAME == 'fundamental':
        n = create_fundamental_jsonl(TRAIN_JSONL)
    else:
        n = create_market_sentiment_jsonl(TRAIN_JSONL)
    print('Created examples:', n)
    print('Saved:', TRAIN_JSONL)

## 3. Load JSONL Messages

In [ ]:
def load_messages(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line)['messages'])
    return rows

messages = load_messages(TRAIN_JSONL)
print('Training examples:', len(messages))
print(json.dumps(messages[0], indent=2)[:1200])

## 4. Load Tokenizer And Format Chat Text

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_messages(chat_messages):
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            chat_messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    chunks = []
    for msg in chat_messages:
        chunks.append(f"{msg['role'].upper()}: {msg['content']}")
    return '\n\n'.join(chunks) + tokenizer.eos_token

texts = [format_messages(row) for row in messages]
dataset = Dataset.from_dict({'text': texts})
print(dataset)
print(texts[0][:1200])

## 5. Tokenize Dataset

In [ ]:
def tokenize(batch):
    tokenized = tokenizer(
        batch['text'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
    )
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])
print(tokenized_dataset)

## 6. Load Base Model And Attach LoRA Adapter

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map='auto' if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Train

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
    fp16=torch.cuda.is_available() and dtype == torch.float16,
    bf16=torch.cuda.is_available() and dtype == torch.bfloat16,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

## 8. Save LoRA Adapter

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved LoRA adapter to:', OUTPUT_DIR)

## 9. Test Inference With Fine-Tuned Adapter

In [ ]:
from peft import PeftModel

test_prompt = messages[0][1]['content']
test_messages = [
    {'role': 'system', 'content': 'You are a financial risk agent. Return strict JSON.'},
    {'role': 'user', 'content': test_prompt},
]

if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
    prompt_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
else:
    prompt_text = f"SYSTEM: {test_messages[0]['content']}\n\nUSER: {test_messages[1]['content']}\n\nASSISTANT:"

inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))